# 12.7 - Scaling AI Apps

**Module 12 - Production Deployment** | Netsetos GenAI Engineering

This notebook builds the scaling toolkit for an AI app as seven tiny runnable models: why LLM apps are I/O-bound and must scale on concurrency (not CPU), horizontal-vs-vertical and statelessness, concurrency-based autoscaling, cold starts with min/max instances, queue-based load leveling, GPU batching with vLLM, and backpressure at the hard ceiling. Everything runs with no cloud, no GPU, and no API key - each cell is a small decision function you can read and re-run.

Read top to bottom: each code cell has a short **intro above** it and a **step-by-step walkthrough below** it. Run the cell, then check its output against the walkthrough.

## Setup

Install the one optional dependency this lesson uses. Every code cell after this is pure Python standard library - no SDKs, no cloud calls - so this install is only relevant if you want to load keys from a `.env` file on a fresh machine or Colab.

In [ ]:
# Install Python dependencies used by this lesson (uncomment on Colab / fresh env).
# !pip install python-dotenv -q  # noqa


**Explanation**

A dependency-install cell, commented out by default. `python-dotenv` is the only third-party package the lesson references, and only for optionally reading keys from a `.env` file; none of the runnable models here actually need it.

**How the code works - step by step**
- **`!pip install python-dotenv -q`** - commented out; uncomment on Colab or a fresh environment to enable `.env`-file key loading.
- The `# noqa` marks the shell magic so linters leave it alone.

**In one line:** optional install of `python-dotenv`; nothing here is required to run the models.

**What you'll see:** (no output - environment prep)

## Environment keys (optional)

Set up placeholder environment variables for any provider keys. None of the seven scaling models call a real API, so you can run the whole notebook with these left blank - the keys are here only so the patterns match a real service.

In [ ]:
import os
# Load API keys from the environment (or a .env file via python-dotenv).
# Never hardcode keys. Any ONE provider key is enough to start;
# the multi-provider demos are best with two or more.
os.environ.setdefault("OPENAI_API_KEY", "")     # platform.openai.com
os.environ.setdefault("ANTHROPIC_API_KEY", "")    # console.anthropic.com
os.environ.setdefault("GOOGLE_API_KEY", "")     # aistudio.google.com


**Explanation**

A key-loading cell that uses `os.environ.setdefault` so it never overwrites a key you already exported. It registers empty defaults for three providers; because every model in this notebook is keyless, this cell is purely for parity with a production service.

**How the code works - step by step**
- **`import os`** - the standard-library module used to read and set environment variables.
- **`os.environ.setdefault("OPENAI_API_KEY", "")`** - registers an empty default only if the variable is unset (an already-exported key wins).
- **Same for `ANTHROPIC_API_KEY` and `GOOGLE_API_KEY`** - the comment on each names where to get it.

**In one line:** register empty, non-destructive placeholders for provider keys you will not actually need here.

**What you'll see:** (no output - environment prep)

## Reproducibility note

A one-line marker cell. The lesson's models use fixed constants rather than live randomness, so there is nothing to seed - this cell just flags the intent.

In [ ]:
# Reproducibility - the lesson uses seeded random values throughout

**Explanation**

A comment-only cell with no executable statements. It documents that the notebook favours seeded/fixed values for repeatable output; in practice every model below is deterministic (fixed constants, no random calls), so running it changes nothing.

**How the code works - step by step**
- **The single comment line** - states that the lesson uses seeded random values so results are reproducible.

**In one line:** a documentation marker; nothing executes.

**What you'll see:** No output - it is a comment cell with no print statements.

## 1 - Why AI apps scale differently: CPU stays low while you drown

A traditional web app is CPU-bound, so watching CPU tells you when to add instances. An LLM app that calls a model API is the opposite - it spends each request **waiting** on the network, so an instance can be completely saturated while CPU reads 30 percent. This cell shows a CPU autoscaler sleeping through a spike, and why concurrency is the signal that actually reflects load.

In [ ]:
# An LLM-API instance spends most of each request WAITING on the model (I/O-bound),
# so CPU stays LOW even when the instance is saturated with in-flight requests.
CPU_PER_INFLIGHT = 0.001    # a waiting request barely uses CPU
CPU_TARGET = 0.60           # a CPU autoscaler fires above 60%
CONCURRENCY_TARGET = 80     # a concurrency autoscaler fires above 80 in-flight per instance

def decide(inflight):
    cpu = min(1.0, inflight * CPU_PER_INFLIGHT)
    cpu_scaler = "SCALE UP" if cpu > CPU_TARGET else "asleep (cpu below 60%)"
    conc_scaler = "SCALE UP" if inflight > CONCURRENCY_TARGET else "hold"
    return cpu, cpu_scaler, conc_scaler

print("An I/O-bound LLM app under a traffic spike (in-flight requests per instance):")
for inflight in [40, 120, 300]:
    cpu, cs, cc = decide(inflight)
    print("  in-flight {:>3}:  CPU {:>2.0f}%   CPU-autoscaler: {:<22}  concurrency-autoscaler: {}".format(
        inflight, cpu * 100, cs, cc))
print()
print("At 300 in-flight the instance is drowning, but CPU is only 30%: a CPU autoscaler never scales up.")
print("Scale an I/O-bound LLM app on CONCURRENCY (in-flight requests), not CPU.")

# Output:
# An I/O-bound LLM app under a traffic spike (in-flight requests per instance):
#   in-flight  40:  CPU  4%   CPU-autoscaler: asleep (cpu below 60%)  concurrency-autoscaler: hold
#   in-flight 120:  CPU 12%   CPU-autoscaler: asleep (cpu below 60%)  concurrency-autoscaler: SCALE UP
#   in-flight 300:  CPU 30%   CPU-autoscaler: asleep (cpu below 60%)  concurrency-autoscaler: SCALE UP
#
# At 300 in-flight the instance is drowning, but CPU is only 30%: a CPU autoscaler never scales up.
# Scale an I/O-bound LLM app on CONCURRENCY (in-flight requests), not CPU.

**Explanation**

A side-by-side comparison harness, not a model call. It models an I/O-bound instance where each in-flight request costs almost no CPU, then asks two autoscalers - one watching CPU, one watching in-flight count - what they would do at increasing load. The point is to make the CPU autoscaler visibly fail to fire while the instance drowns.

**How the code works - step by step**
- **`CPU_PER_INFLIGHT`, `CPU_TARGET`, `CONCURRENCY_TARGET`** - constants encoding that a waiting request barely uses CPU (0.001), a CPU scaler fires above 60%, and a concurrency scaler fires above 80 in-flight.
- **`decide(inflight)`** - computes CPU as `inflight * CPU_PER_INFLIGHT` (capped at 1.0), then returns what each autoscaler decides at that load.
- **The loop over `[40, 120, 300]`** - runs the same instance through three levels of a traffic spike and prints CPU% and both scaler verdicts.

**In one line:** climb the in-flight count, watch CPU stay flat, watch only the concurrency scaler react.

**What you'll see:** A three-row table: at 40 in-flight CPU is 4% (both scalers idle), at 120 CPU is 12% (CPU asleep, concurrency scales up), at 300 CPU is only 30% (CPU still asleep, concurrency scales up) - with the takeaway that an I/O-bound app must scale on concurrency, not CPU.

## 2 - Horizontal vs vertical, and why statelessness is the catch

Two ways to add capacity: make one instance bigger (vertical) or add more instances behind a load balancer (horizontal). Horizontal wins for the web tier - linear capacity, no single point of failure - but only if instances are **stateless**, so any instance can serve any request. This cell prices both directions and then demonstrates the session-loss bug that local state causes.

In [ ]:
# Vertical = a bigger instance. Horizontal = more instances. Horizontal needs STATELESS.
def capacity(instances, per_instance_rps):
    return instances * per_instance_rps

print("Scaling a web/API tier - two directions:")
print("  vertical   (1 bigger instance):  {:>4} req/s  -> ceiling = the biggest machine; a single point of failure".format(capacity(1, 500)))
print("  horizontal (10 instances):       {:>4} req/s  -> linear capacity, no single point of failure".format(capacity(10, 100)))
print()

# Horizontal only works if instances are STATELESS: any instance can serve any request.
def serve(req_instance, session_home, store):
    if store == "local":
        return "OK" if req_instance == session_home else "FAIL: session is on a different instance"
    return "OK: session lives in a shared store, so any instance serves it"

print("Horizontal scaling requires STATELESS instances:")
print("  local session,  request routed to a DIFFERENT instance  -> " + serve("B", "A", "local"))
print("  shared session, request routed to a DIFFERENT instance  -> " + serve("B", "A", "shared"))
print()
print("Scale the STATELESS web/API tier OUT (horizontal); scale the model tier UP (bigger GPU) or out (more replicas).")

# Output:
# Scaling a web/API tier - two directions:
#   vertical   (1 bigger instance):   500 req/s  -> ceiling = the biggest machine; a single point of failure
#   horizontal (10 instances):       1000 req/s  -> linear capacity, no single point of failure
#
# Horizontal scaling requires STATELESS instances:
#   local session,  request routed to a DIFFERENT instance  -> FAIL: session is on a different instance
#   shared session, request routed to a DIFFERENT instance  -> OK: session lives in a shared store, so any instance serves it
#
# Scale the STATELESS web/API tier OUT (horizontal); scale the model tier UP (bigger GPU) or out (more replicas).

**Explanation**

A two-part demonstration. The first part is a trivial capacity calculator contrasting one big instance against ten small ones; the second is a routing simulator showing what happens when a request lands on a different instance than the one holding its session. Together they make the stateless requirement concrete rather than abstract.

**How the code works - step by step**
- **`capacity(instances, per_instance_rps)`** - multiplies to show vertical (1x500) vs horizontal (10x100) throughput.
- **`serve(req_instance, session_home, store)`** - if state is `local`, it only succeeds when the request lands on the session's home instance; if `shared`, any instance succeeds.
- **The two `serve(...)` calls** - both route to a *different* instance than the session home, so `local` fails and `shared` works.

**In one line:** more instances give linear capacity, but only if session state lives in a shared store, not on local disk.

**What you'll see:** Vertical shows 500 req/s (a single point of failure), horizontal shows 1000 req/s; then the local-session route FAILs on a different instance while the shared-store route is OK - closing with scale the web tier out, the model tier up or via replicas.

## 3 - Autoscale on concurrency, tuned to the workload

Cloud Run scales on request concurrency, targeting roughly 60 percent of the max you set with `--concurrency`. The right value depends entirely on the workload: **high** (80-200) for an I/O-bound app that calls an external API, **low** (1-4) for a compute-bound model running on the instance. This cell shows how that single setting swings your fleet size by 20x - and how getting it backwards breaks both cases.

In [ ]:
# Cloud Run scales on request CONCURRENCY (target ~60% of the max you set).
# The right --concurrency depends on whether the app WAITS (I/O) or COMPUTES.
WORKLOADS = {
    "calls an LLM API (I/O-bound)":        {"concurrency": 80, "note": "one instance holds many waiting requests"},
    "runs a model on-instance (compute)":  {"concurrency": 4,  "note": "each request saturates the CPU/GPU"},
}
def instances_needed(concurrent, per_instance):
    return -(-concurrent // per_instance)   # ceiling division

print("Serving 240 concurrent requests - set --concurrency by workload:")
for name, w in WORKLOADS.items():
    n = instances_needed(240, w["concurrency"])
    print("  {:<34} --concurrency={:<3} -> {:>3} instances   ({})".format(name, w["concurrency"], n, w["note"]))
print()
print("Setting it wrong is the classic AI scaling bug:")
print("  I/O-bound app with --concurrency=1   -> {} instances for 240 waiting requests (huge waste)".format(instances_needed(240, 1)))
print("  compute-bound app with --concurrency=80 -> 80 requests fight over one GPU (all slow, or out of memory)")
print()
print("Set concurrency HIGH for an API-calling (I/O-bound) app, LOW for an on-instance model.")

# Output:
# Serving 240 concurrent requests - set --concurrency by workload:
#   calls an LLM API (I/O-bound)       --concurrency=80  ->   3 instances   (one instance holds many waiting requests)
#   runs a model on-instance (compute) --concurrency=4   ->  60 instances   (each request saturates the CPU/GPU)
#
# Setting it wrong is the classic AI scaling bug:
#   I/O-bound app with --concurrency=1   -> 240 instances for 240 waiting requests (huge waste)
#   compute-bound app with --concurrency=80 -> 80 requests fight over one GPU (all slow, or out of memory)
#
# Set concurrency HIGH for an API-calling (I/O-bound) app, LOW for an on-instance model.

**Explanation**

A fleet-sizing calculator. It sets two workload profiles with opposite concurrency needs, computes how many instances each needs to serve the same 240 concurrent requests, then deliberately swaps the settings to show the two classic failure modes. It is a what-if harness for one deploy flag.

**How the code works - step by step**
- **`WORKLOADS`** - a dict of two profiles: I/O-bound at concurrency 80, compute-bound at concurrency 4, each with a one-line reason.
- **`instances_needed(concurrent, per_instance)`** - ceiling division (`-(-a // b)`) to get whole instances.
- **The loop over `WORKLOADS`** - sizes the fleet for 240 concurrent under each profile.
- **The two mis-set lines** - concurrency 1 on the I/O app (240 instances, huge waste) and concurrency 80 on the model (80 requests fighting one GPU).

**In one line:** match `--concurrency` to whether the app waits or computes, or you either waste money or melt the GPU.

**What you'll see:** The I/O-bound app needs just 3 instances at concurrency 80, the compute-bound model needs 60 at concurrency 4; the mis-set section shows 240 instances for the I/O app at concurrency 1 and a GPU-contention warning for the model at concurrency 80.

## 4 - Cold starts, min-instances and max-instances

Two dials bound the fleet. Scale-to-zero costs nothing idle but makes the first request pay a cold start - and for an AI service loading a model that is 10-40 seconds, not milliseconds. Min-instances keeps warm capacity so nobody pays it; max-instances caps cost and, more importantly for an AI app, protects the downstream provider's rate limit. This cell models both dials.

In [ ]:
# Scale-to-zero saves money but the first request after idle pays a COLD START.
COLD_START_S = 30    # loading the model / heavy deps on a fresh instance
WARM_S = 0.2

print("Cold starts - the first request after the service scaled to zero:")
print("  --min-instances=0 (scale to zero): first request waits ~{:.0f}s (cold start), then fast; no idle cost".format(COLD_START_S))
print("  --min-instances=1 (keep 1 warm):   first request ~{:.1f}s (no cold start); a small idle cost".format(WARM_S))
print()

# --max-instances caps cost AND protects the downstream provider's rate limit.
PROVIDER_TPM = 200000                 # tokens/min the provider allows
TOKENS_PER_INSTANCE_PER_MIN = 20000   # tokens/min one instance drives
safe_max = PROVIDER_TPM // TOKENS_PER_INSTANCE_PER_MIN

print("--max-instances protects the downstream provider (and caps cost):")
print("  provider allows {:,} tokens/min; each instance drives ~{:,} tokens/min".format(PROVIDER_TPM, TOKENS_PER_INSTANCE_PER_MIN))
print("  so --max-instances={} keeps the whole fleet under the provider's rate limit".format(safe_max))
print("  scaling past that just returns 429s from the provider - more instances do NOT help.")

# Output:
# Cold starts - the first request after the service scaled to zero:
#   --min-instances=0 (scale to zero): first request waits ~30s (cold start), then fast; no idle cost
#   --min-instances=1 (keep 1 warm):   first request ~0.2s (no cold start); a small idle cost
#
# --max-instances protects the downstream provider (and caps cost):
#   provider allows 200,000 tokens/min; each instance drives ~20,000 tokens/min
#   so --max-instances=10 keeps the whole fleet under the provider's rate limit
#   scaling past that just returns 429s from the provider - more instances do NOT help.

**Explanation**

Two small calculators bundled in one cell. The first contrasts cold-start latency against a warm instance to justify min-instances; the second derives a safe max-instances value from the provider's token-per-minute limit divided by what one instance drives. The second is the load-bearing idea: max-instances is a rate-limit guard, not just a budget cap.

**How the code works - step by step**
- **`COLD_START_S`, `WARM_S`** - 30s to load a model on a fresh instance vs 0.2s when warm.
- **The two min-instances lines** - `--min-instances=0` pays the cold start once; `--min-instances=1` keeps a warm instance for a small idle cost.
- **`PROVIDER_TPM`, `TOKENS_PER_INSTANCE_PER_MIN`, `safe_max`** - divides the provider's 200k TPM by the 20k one instance drives to get a safe max-instances of 10.
- **The max-instances lines** - explain that scaling past `safe_max` only collects 429s.

**In one line:** keep one warm to kill cold starts, and cap the fleet at the provider's rate limit so extra instances do not just earn 429s.

**What you'll see:** The cold-start block prints ~30s first request at min-instances=0 vs ~0.2s at min-instances=1; the provider block computes --max-instances=10 from 200,000 TPM / 20,000 per instance, and notes scaling past that returns 429s.

## 5 - Queue-based load leveling: absorb the spike, drop nothing

Autoscaling reacts too slowly for some bursts. Instead of dropping the overflow, enqueue it: the front end writes each job to a queue and returns immediately, workers drain it at a steady rate, and you autoscale the workers on **queue depth** (this is what KEDA does). This cell runs a 500-request burst through a capacity-100 tier, first without a queue, then with one, and drains the backlog tick by tick.

In [ ]:
# A burst above capacity: drop it, or QUEUE it and drain at a steady rate.
BURST = 500           # requests arriving in one spike
CAPACITY = 100        # requests the synchronous tier can handle at once

print("A burst of {} requests hits a tier that can handle {} at once:".format(BURST, CAPACITY))
overflow = max(0, BURST - CAPACITY)
print("  WITHOUT a queue: {} served now, {} OVERFLOW -> dropped / timed out".format(CAPACITY, overflow))
print()

# WITH a queue: enqueue everything; autoscale workers on QUEUE DEPTH (not CPU).
TARGET_PER_WORKER = 50    # aim for ~50 queued items per worker
MAX_WORKERS = 4          # maxReplicaCount - caps cost and protects downstream
def workers_for(depth):
    return min(MAX_WORKERS, -(-depth // TARGET_PER_WORKER))

print("  WITH a queue (autoscale workers on queue DEPTH, capped at {} workers):".format(MAX_WORKERS))
depth, tick = BURST, 0
while depth > 0 and tick < 8:
    w = workers_for(depth)
    drained = min(depth, w * TARGET_PER_WORKER)
    print("    tick {}: depth {:>3} -> {} workers -> drain {} -> depth {}".format(tick, depth, w, drained, depth - drained))
    depth -= drained
    tick += 1
print("  the queue ABSORBS the spike: nothing is dropped, the work is just delayed a little.")

# Output:
# A burst of 500 requests hits a tier that can handle 100 at once:
#   WITHOUT a queue: 100 served now, 400 OVERFLOW -> dropped / timed out
#
#   WITH a queue (autoscale workers on queue DEPTH, capped at 4 workers):
#     tick 0: depth 500 -> 4 workers -> drain 200 -> depth 300
#     tick 1: depth 300 -> 4 workers -> drain 200 -> depth 100
#     tick 2: depth 100 -> 2 workers -> drain 100 -> depth 0
#   the queue ABSORBS the spike: nothing is dropped, the work is just delayed a little.

**Explanation**

A small simulation loop, not a real queue. It first shows the naive overflow (burst minus capacity is dropped), then models a capped worker pool draining the backlog over several ticks, with worker count itself scaled on remaining depth. The loop is the point: it visualizes the queue decoupling arrival rate from processing rate.

**How the code works - step by step**
- **`BURST`, `CAPACITY`** - 500 arriving vs 100 the synchronous tier can serve at once.
- **`overflow = max(0, BURST - CAPACITY)`** - the 400 that get dropped with no queue.
- **`TARGET_PER_WORKER`, `MAX_WORKERS`, `workers_for(depth)`** - aim for ~50 queued per worker, capped at 4 (the KEDA `maxReplicaCount`), via ceiling division.
- **The `while depth > 0` loop** - each tick recomputes workers from the current depth, drains `workers * 50`, and prints the shrinking backlog.

**In one line:** enqueue everything, autoscale workers on backlog depth, and drain the spike over a few ticks instead of dropping it.

**What you'll see:** Without a queue: 100 served, 400 overflow dropped. With a queue: depth falls 500 -> 300 -> 100 -> 0 over three ticks (4 workers, then 2 as the backlog shrinks), and nothing is dropped.

## 6 - Scaling the model tier: GPU batching with vLLM

If you host the model yourself, throughput comes from **batching** requests together. A naive one-at-a-time loop leaves the GPU around 30 percent utilized; vLLM's continuous batching packs many requests per token-step to lift it to ~85 percent, and PagedAttention pages the KV cache to fit 2-3x more concurrent requests. Together that is roughly 3-5x throughput on the identical GPU. This cell compares the two.

In [ ]:
# A self-hosted model on one GPU. Throughput comes from BATCHING requests together.
def gpu(mode):
    if mode == "naive":     # one request at a time
        return {"util": 30, "concurrent": 8,  "tok_per_s": 2600}
    return {"util": 85, "concurrent": 24, "tok_per_s": 9000}   # continuous batching + PagedAttention

naive, batched = gpu("naive"), gpu("batched")
print("One GPU, self-hosted model - naive loop vs continuous batching (vLLM):")
print("  {:<32} GPU {:>2}%   {:>2} concurrent   {:,} tokens/s".format("naive (one request at a time)", naive["util"], naive["concurrent"], naive["tok_per_s"]))
print("  {:<32} GPU {:>2}%   {:>2} concurrent   {:,} tokens/s".format("continuous batching (vLLM)", batched["util"], batched["concurrent"], batched["tok_per_s"]))
print()
print("  batching lifts GPU utilization {}% -> {}% and throughput ~{:.1f}x on the SAME GPU.".format(
    naive["util"], batched["util"], batched["tok_per_s"] / naive["tok_per_s"]))
print("  PagedAttention pages the KV cache, so memory waste drops from ~60% to near zero (2-3x more concurrent).")
print("  Then autoscale GPU REPLICAS on queue depth, and scale to zero when idle.")

# Output:
# One GPU, self-hosted model - naive loop vs continuous batching (vLLM):
#   naive (one request at a time)    GPU 30%    8 concurrent   2,600 tokens/s
#   continuous batching (vLLM)       GPU 85%   24 concurrent   9,000 tokens/s
#
#   batching lifts GPU utilization 30% -> 85% and throughput ~3.5x on the SAME GPU.
#   PagedAttention pages the KV cache, so memory waste drops from ~60% to near zero (2-3x more concurrent).
#   Then autoscale GPU REPLICAS on queue depth, and scale to zero when idle.

**Explanation**

A lookup-and-compare harness. It returns fixed utilization/concurrency/throughput numbers for a naive loop vs a batched (vLLM) server, then computes the throughput ratio between them. There is no actual GPU or model here - it is an illustrative before/after that quantifies why batching, not bigger hardware, is the lever.

**How the code works - step by step**
- **`gpu(mode)`** - returns a dict of `util`, `concurrent`, `tok_per_s`: naive (30%, 8, 2600) vs batched (85%, 24, 9000).
- **`naive, batched = gpu("naive"), gpu("batched")`** - grabs both profiles.
- **The two print rows** - lay the profiles side by side.
- **`batched["tok_per_s"] / naive["tok_per_s"]`** - computes the ~3.5x throughput gain, followed by notes on PagedAttention and autoscaling replicas on queue depth.

**In one line:** continuous batching plus PagedAttention lifts a single GPU from 30% to 85% utilization for ~3.5x throughput.

**What you'll see:** Two rows - naive at 30% GPU / 8 concurrent / 2,600 tokens/s vs vLLM at 85% / 24 concurrent / 9,000 tokens/s - and the computed ~3.5x throughput gain on the same GPU, plus the note to autoscale GPU replicas on queue depth.

## 7 - The scaling ceiling: backpressure

Autoscaling, queues, and batching all push the ceiling up, but they never remove it - there is always a hard cap (the provider's rate limit or your GPU fleet). Past it, adding instances only collects 429s. The last piece of a scaling design is what to do *at* the ceiling: shed the excess deliberately with a 429 + Retry-After or a degraded reply, so accepted requests stay fast instead of everyone timing out. This cell contrasts collapse against graceful shedding.

In [ ]:
# Autoscaling + queues + batching raise the ceiling, but there is ALWAYS a hard ceiling
# (the provider rate limit / your GPU fleet cap). Past it, more instances do not help.
CEILING = 500    # req/s the whole system can actually serve

print("Demand climbs toward the hard ceiling of {} req/s (provider rate limit / GPU cap):".format(CEILING))
for demand in [300, 500, 800]:
    if demand <= CEILING:
        print("  demand {:>3}: served {}, under the ceiling - fine".format(demand, demand))
        continue
    excess = demand - CEILING
    print("  demand {:>3}: over the ceiling by {}".format(demand, excess))
    print("      no backpressure  -> the overload times out ALL {} requests (collapse)".format(demand))
    print("      with backpressure -> serve {} fast, shed {} with 429 + Retry-After (or a cached/degraded reply)".format(CEILING, excess))
print()
print("You cannot scale past the ceiling. When you hit it, SHED load gracefully instead of collapsing")
print("(the retry / degradation mechanics are Lesson 12.2; the rate-limit config is Lesson 12.3).")

# Output:
# Demand climbs toward the hard ceiling of 500 req/s (provider rate limit / GPU cap):
#   demand 300: served 300, under the ceiling - fine
#   demand 500: served 500, under the ceiling - fine
#   demand 800: over the ceiling by 300
#       no backpressure  -> the overload times out ALL 800 requests (collapse)
#       with backpressure -> serve 500 fast, shed 300 with 429 + Retry-After (or a cached/degraded reply)
#
# You cannot scale past the ceiling. When you hit it, SHED load gracefully instead of collapsing
# (the retry / degradation mechanics are Lesson 12.2; the rate-limit config is Lesson 12.3).

**Explanation**

A decision harness over a fixed ceiling. For each demand level it checks whether demand is under the cap, and if over, prints the two outcomes: no backpressure (the whole overloaded system times out) vs backpressure (serve the cap fast, shed the excess honestly). It reframes overload as a choice about *how* to fail, not whether to.

**How the code works - step by step**
- **`CEILING`** - 500 req/s the whole system can actually serve.
- **The loop over `[300, 500, 800]`** - for demand at or below the ceiling it prints served-fine; above it, it computes `excess = demand - CEILING`.
- **The two over-ceiling branches** - no backpressure times out all 800 (collapse); with backpressure it serves 500 fast and sheds 300 with 429 + Retry-After or a cached/degraded reply.

**In one line:** you cannot scale past the hard ceiling, so at it, shed the excess gracefully rather than letting everything collapse.

**What you'll see:** Demand 300 and 500 are served under the ceiling; at 800 (over by 300), no-backpressure times out all 800 while with-backpressure serves 500 fast and sheds 300 with a 429 - pointing to Lesson 12.2 for the retry/degradation mechanics.

## 8 - The whole design in one config file

A single illustrative config that puts all seven pieces together across three tiers: the stateless web/API tier on Cloud Run, the queue-worker tier on KEDA, and the self-hosted model tier on vLLM. Each tier scales on its *own* right signal - concurrency, queue depth, and batch fullness - which is the entire lesson expressed as deploy flags.

> **Illustrative only** - needs gcloud, a KEDA-enabled cluster, and a GPU; not run locally.

In [ ]:
# SCALING CONFIG - an illustrative minimal example (Cloud Run + KEDA + vLLM).

# --- The stateless web/API tier: autoscale on CONCURRENCY, keep 1 warm, cap the fleet ---
#   gcloud run deploy ai-svc \
#     --concurrency=80 \            # I/O-bound (calls an LLM API): one instance holds many waiting requests
#     --min-instances=1 \          # keep 1 warm -> no cold start on the first request
#     --max-instances=10 \         # cap the fleet under the provider rate limit (200k TPM / 20k per instance)
#     --cpu-boost                  # faster cold start when it does scale up

# --- The queue-worker tier: autoscale on QUEUE DEPTH with KEDA (Kubernetes), scaledobject.yaml ---
#   apiVersion: keda.sh/v1alpha1
#   kind: ScaledObject
#   spec:
#     scaleTargetRef: {name: llm-worker}
#     minReplicaCount: 0           # scale to zero when the queue is empty
#     maxReplicaCount: 4           # protect downstream; cap cost
#     triggers:
#       - type: gcp-pubsub          # scale on the Pub/Sub backlog, not CPU
#         metadata: {subscriptionName: jobs-sub, mode: NumUndeliveredMessages, value: "50"}   # ~50 queued per worker

# --- The model tier: vLLM for continuous batching + PagedAttention on one GPU ---
#   vllm serve meta-llama/Llama-3.1-8B-Instruct \
#     --max-num-seqs 24 \          # continuous batching: up to 24 requests packed per step
#     --gpu-memory-utilization 0.9 # PagedAttention packs the KV cache tightly
# Output: (illustrative minimal example - needs gcloud, a KEDA-enabled cluster, and a GPU; not run locally.)

**Explanation**

A reference config cell, not runnable code - every line is a comment. It is the capstone artifact that maps each earlier model to a real production knob: the concurrency/min/max flags from Steps 3-4, the KEDA queue-depth trigger from Step 5, and the vLLM batching flags from Step 6, all in one place.

**How the code works - step by step**
- **The `gcloud run deploy` block** - the web tier: `--concurrency=80` (I/O-bound), `--min-instances=1` (no cold start), `--max-instances=10` (under the provider ceiling), `--cpu-boost`.
- **The KEDA `ScaledObject` block** - the queue-worker tier: `minReplicaCount: 0` (scale to zero), `maxReplicaCount: 4`, and a `gcp-pubsub` trigger on `NumUndeliveredMessages` (queue depth, ~50 per worker).
- **The `vllm serve` block** - the model tier: `--max-num-seqs 24` (continuous batching) and `--gpu-memory-utilization 0.9` (PagedAttention).

**In one line:** three tiers, each autoscaling on the signal that fits it - concurrency, queue depth, batch fullness.

**What you'll see:** No live output - the closing comment states it is an illustrative minimal example needing gcloud, a KEDA-enabled cluster, and a GPU, and is not run locally.

Across seven runnable models you scaled every tier of an AI app on its own right signal: the web tier on concurrency (high for I/O-bound, low for compute-bound), kept warm with min-instances and capped under the provider with max-instances; the queue-worker tier on backlog depth; the model tier on GPU batch fullness via vLLM; and, at the unavoidable hard ceiling, backpressure that sheds excess with a 429 instead of collapsing. The five questions to ask any scaling design - right signal, stateless, warm, capped to the provider, buffered by a queue, graceful at the ceiling - fall straight out of these cells. Next: Lesson 12.8 monitors these same signals (RPS, concurrency, queue depth, p95, GPU util), Lesson 12.2 covers the retry/degradation mechanics behind backpressure, and Module 13 handles the economics of scale.